In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
df_model = df_new.drop(columns=['date', 'price_per_sqft']).copy()
df_model['zipcode'] = df_model['zipcode'].astype(str)
y = df_model['price']
X = df_model.drop(columns=['price'])

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size = 0.2, random_state = 10)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size = 0.2, random_state = 10)

In [ ]:
X_train.shape

In [ ]:
X_val.shape

In [ ]:
X_test.shape

In [ ]:
categorical = list(X_train.select_dtypes(include=['object', 'category', 'string']).columns)
numeric_features = list(X_train.select_dtypes(include=[np.number]).columns)
print('Категориальные:', categorical)
print('Числовых признаков:', len(numeric_features))

In [ ]:
numeric_pipe = Pipeline(steps=[('scaler', StandardScaler())])

preprocessor = ColumnTransformer([('ohe', OneHotEncoder(handle_unknown='ignore'), categorical), ('num', numeric_pipe, numeric_features)])

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

for name in ['X_train_proc', 'X_val_proc', 'X_test_proc']:
    arr = eval(name)
    if hasattr(arr, 'toarray'):
        arr = arr.toarray()
    globals()[name] = arr.astype('float32')

In [ ]:
input_dim = X_train_proc.shape[1]
input_dim

In [ ]:
y_train_log = np.log1p(y_train.values)
y_val_log = np.log1p(y_val.values)

y_mean = y_train_log.mean()
y_std = y_train_log.std()

y_train_scaled = ((y_train_log - y_mean) / y_std).astype('float32')


def to_price(pred_scaled):
    pred_log = pred_scaled * y_std + y_mean
    return np.expm1(pred_log)

In [ ]:
X_train_t = torch.tensor(X_train_proc)
y_train_t = torch.tensor(y_train_scaled).view(-1, 1)
X_val_t = torch.tensor(X_val_proc)
X_test_t = torch.tensor(X_test_proc)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

Четыре полносвязные сети:
MLP_small
MLP_medium
MLP_deep
ResidualMLP

Простая сеть: один скрытый слой, ReLU, без регуляризации

In [ ]:
class MLP_small(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1))

    def forward(self, x):
        x = self.net(x)
        return x

Средняя сеть: два скрытых слоя, ReLU, BatchNorm + Dropout

In [ ]:
class MLP_medium(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 1))

    def forward(self, x):
        x = self.net(x)
        return x

Глубокая сеть: 4 скрытых слоя, GELU, BatchNorm + Dropout

In [ ]:
class MLP_deep(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Linear(64, 1))

    def forward(self, x):
        x = self.net(x)
        return x

Остаточный блок: два полносвязных с BatchNorm и Dropout, активация GELU, со skip-связью

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(width, width),
            nn.BatchNorm1d(width),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(width, width),
            nn.BatchNorm1d(width))
        self.activation = nn.GELU()

    def forward(self, x):
        x = self.activation(x + self.net(x))
        return x

Остаточная полносвязная сеть: вход, 3 остаточных блока, GELU, Dropout

In [ ]:
class ResidualMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        width = 256
        n_blocks = 3

        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.BatchNorm1d(width),
            nn.GELU(),
            nn.Dropout(0.2))

        self.res_blocks = nn.Sequential(*[ResidualBlock(width) for _ in range(n_blocks)])

        self.output_layer = nn.Linear(width, 1)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.res_blocks(x)
        x = self.output_layer(x)
        return x

Настроим MLflow

In [ ]:
!pip install -q mlflow joblib

In [ ]:
import mlflow
import joblib
import json
from pathlib import Path

mlflow.end_run()

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    storage_root = Path('/content/drive/MyDrive')
except ImportError:
    storage_root = Path.cwd()

In [ ]:
project_dir = storage_root / 'GP5_DL_real_estate_tabular'
mlflow_dir = project_dir / 'mlflow'
mlflow_artifacts_dir = mlflow_dir / 'artifacts'
models_dir = project_dir / 'models'
data_artifacts_dir = project_dir / 'data_artifacts'
preprocessing_dir = project_dir / 'preprocessing'

In [ ]:
for folder in [project_dir, mlflow_dir, mlflow_artifacts_dir, models_dir, data_artifacts_dir, preprocessing_dir]:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
mlflow_db_path = mlflow_dir / 'mlflow.db'
experiment_name = 'kc_house_price_mlp'

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{mlflow_db_path.as_posix()}")

In [ ]:
experiment = mlflow.get_experiment_by_name(experiment_name)

In [ ]:
if experiment is None:
    experiment_id = mlflow.create_experiment(name = experiment_name, artifact_location=mlflow_artifacts_dir.resolve().as_uri())
else:
    experiment_id = experiment.experiment_id

In [ ]:
mlflow.set_experiment(experiment_name)

In [ ]:
print('MLflow подключен к ', mlflow.get_tracking_uri())
print('Куда сохраняються модели:', mlflow_artifacts_dir)

In [ ]:
data_path = data_artifacts_dir / 'train_val_test.npz'
preprocessor_path = preprocessing_dir / 'preprocessor.joblib'

np.savez_compressed(
    data_path,
    X_train=X_train_proc,
    X_val=X_val_proc,
    X_test=X_test_proc,
    y_train=y_train.values,
    y_val=y_val.values,
    y_test=y_test.values,
    y_mean=y_mean,
    y_std=y_std)

joblib.dump(preprocessor, preprocessor_path)

Создаем функции

In [ ]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score


def fit_epoch(model, train_loader, criterion, optimizer, device=device):
    model.train()
    train_losses = []

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = criterion(model(data), target)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    return float(np.mean(train_losses))

In [ ]:
def eval_epoch(model, X_val_t, y_val_log, device=device):
    model.eval()

    with torch.no_grad():
        val_pred = model(X_val_t.to(device)).cpu().numpy().ravel()

    return root_mean_squared_error(np.expm1(y_val_log), to_price(val_pred))

In [ ]:
def predict(model, X, device=device):
    model.eval()

    with torch.no_grad():
        pred_scaled = model(X.to(device)).cpu().numpy().ravel()

    return to_price(pred_scaled)

In [ ]:
from tqdm.auto import tqdm


def train(model, name, optimizer, epochs, batch_size, history=None):
    model = model.to(device)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    if history is None:
        history = []

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    best_val_rmse = np.inf
    best_epoch = None
    checkpoint_path = models_dir / f'{name}.pth'

    with mlflow.start_run(run_name=name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            'architecture': model.__class__.__name__,
            'optimizer': optimizer.__class__.__name__,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'weight_decay': optimizer.param_groups[0].get('weight_decay', 0.0),
            'epochs': epochs,
            'batch_size': batch_size,
            'total_params': total_params,
            'trainable_params': trainable_params,
            'n_features': input_dim,
            'seed': 42,
            'target_transform': 'log1p+standardize',
            'dataset_source': 'kc_house_data_NaN.csv',
            'selection_metric': 'validation_rmse',
            'business_task': 'house_price_estimation_for_real_estate_platform'})

        mlflow.log_artifact(str(data_path), artifact_path='data')
        mlflow.log_artifact(str(preprocessor_path), artifact_path='preprocessing')

        with tqdm(desc='epoch', total=epochs, leave=True) as pbar_outer:
            for epoch in range(epochs):
                train_loss = fit_epoch(model, train_loader, criterion, optimizer)
                val_rmse = eval_epoch(model, X_val_t, y_val_log)
                scheduler.step(val_rmse)

                history.append((train_loss, val_rmse))
                mlflow.log_metrics({'train_loss': train_loss, 'val_rmse': val_rmse}, step=epoch + 1)
                pbar_outer.update(1)
                print('Epoch {}: train_loss={:.4f}, val_rmse={:.0f}'.format(epoch + 1, train_loss, val_rmse))

                if val_rmse < best_val_rmse:
                    best_val_rmse = val_rmse
                    best_epoch = epoch + 1
                    checkpoint = {
                        'state_dict': model.state_dict(),
                        'model_class': model.__class__.__name__,
                        'input_dim': input_dim,
                        'y_mean': y_mean,
                        'y_std': y_std,
                        'best_val_rmse': best_val_rmse,
                        'best_epoch': best_epoch}
                    torch.save(checkpoint, checkpoint_path)
                    print('Новая лучшая модель сохранена: val_rmse={:.0f} epoch={}'.format(best_val_rmse, best_epoch))

        mlflow.log_metric('best_val_rmse', best_val_rmse)

        history_df = pd.DataFrame(history, columns=['train_loss', 'val_rmse'])
        history_path = models_dir / f'{name}_train_history.csv'
        history_df.to_csv(history_path, index=False)
        mlflow.log_artifact(str(history_path), artifact_path='history')

        mlflow.log_artifact(str(checkpoint_path), artifact_path='model')

        print('{}: best val RMSE={:.0f} (эпоха {})'.format(name, best_val_rmse, best_epoch))

    return history, run_id, best_val_rmse, best_epoch

Обучение

In [ ]:
results = {}

Обучение MLP_small

In [ ]:
small_epochs = 60
small_batch_size = 256

small_model = MLP_small(input_dim)
small_optimizer = torch.optim.AdamW(small_model.parameters(), lr=1e-3, weight_decay=0.0)

small_history, small_run_id, small_best_val, small_best_epoch = train(model=small_model, name='MLP-small', optimizer=small_optimizer,
                                                                      epochs=small_epochs, batch_size=small_batch_size)

results['MLP-small'] = {'run_id': small_run_id, 'best_val_rmse': small_best_val,
                       'best_epoch': small_best_epoch, 'history': small_history}

Обучение MLP_medium

In [ ]:
medium_epochs = 60
medium_batch_size = 256

medium_model = MLP_medium(input_dim)
medium_optimizer = torch.optim.AdamW(medium_model.parameters(), lr=1e-3, weight_decay=1e-4)

medium_history, medium_run_id, medium_best_val, medium_best_epoch = train(model=medium_model, name='MLP-medium', optimizer=medium_optimizer,
                                                                          epochs=medium_epochs, batch_size=medium_batch_size)

results['MLP-medium'] = {'run_id': medium_run_id, 'best_val_rmse': medium_best_val,
                       'best_epoch': medium_best_epoch, 'history': medium_history}

Обучение MLP_deep

In [ ]:
deep_epochs = 60
deep_batch_size = 256

deep_model = MLP_deep(input_dim)
deep_optimizer = torch.optim.AdamW(deep_model.parameters(), lr=5e-4, weight_decay=1e-4)

deep_history, deep_run_id, deep_best_val, deep_best_epoch = train(model=deep_model, name='MLP-deep', optimizer=deep_optimizer,
                                                                  epochs=deep_epochs, batch_size=deep_batch_size)

results['MLP-deep'] = {'run_id': deep_run_id, 'best_val_rmse': deep_best_val,
                       'best_epoch': deep_best_epoch, 'history': deep_history}

Обучение ResidualMLP

In [ ]:
residual_epochs = 60
residual_batch_size = 256

residual_model = ResidualMLP(input_dim)
residual_optimizer = torch.optim.AdamW(residual_model.parameters(), lr=5e-4, weight_decay=1e-4)

residual_history, residual_run_id, residual_best_val, residual_best_epoch = train(model=residual_model, name='ResidualMLP', optimizer=residual_optimizer,
                                                                                  epochs=residual_epochs, batch_size=residual_batch_size)

results['ResidualMLP'] = {'run_id': residual_run_id, 'best_val_rmse': residual_best_val,
                       'best_epoch': residual_best_epoch, 'history': residual_history}

Достаём из MLflow историю валидационного RMSE

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

plt.figure(figsize=(10, 6))
for name, res in results.items():
    history = sorted(client.get_metric_history(res['run_id'], 'val_rmse'), key = lambda m: m.step)
    plt.plot([m.step for m in history], [m.value for m in history], marker='o', label = name)

plt.xlabel('Эпоха')
plt.ylabel('Validation RMSE')
plt.title('Кривые обучения конфигураций MLP')
plt.legend()
plt.show()

Сраваем модели и выберем лучшую

In [ ]:
comparison = pd.DataFrame({'Model': list(results.keys()),
                           'Validation RMSE': [r['best_val_rmse'] for r in results.values()],
                           'Best epoch': [r['best_epoch'] for r in results.values()]}).sort_values('Validation RMSE').reset_index(drop=True)

comparison

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=comparison, x='Validation RMSE', y='Model')
plt.title('Выбор архитектуры по RMSE')
plt.show()

In [ ]:
best_name = comparison.iloc[0]['Model']
best_run_id = results[best_name]['run_id']
best_name

Тестируем только лучшую

In [ ]:
model_classes = {'MLP_small': MLP_small, 'MLP_medium': MLP_medium, 'MLP_deep': MLP_deep, 'ResidualMLP': ResidualMLP}

best_checkpoint = torch.load(models_dir / f'{best_name}.pth', map_location=device, weights_only=False)
best_model = model_classes[best_checkpoint['model_class']](best_checkpoint['input_dim']).to(device)
best_model.load_state_dict(best_checkpoint['state_dict'])

test_pred = predict(best_model, X_test_t)
test_rmse = root_mean_squared_error(y_test, test_pred)
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metrics({'test_rmse': test_rmse, 'test_mae': test_mae, 'test_r2': test_r2})

print('{}: test RMSE={:.0f}, MAE={:.0f}, R2={:.4f}'.format(best_name, test_rmse, test_mae, test_r2))

Проверим воспроизводимость для задания

In [ ]:
checkpoint = torch.load(models_dir / f'{best_name}.pth', map_location='cpu', weights_only=False)

loaded_model = model_classes[checkpoint['model_class']](checkpoint['input_dim'])
loaded_model.load_state_dict(checkpoint['state_dict'])
loaded_model.eval()

with torch.no_grad():
    pred_scaled = loaded_model(X_test_t.cpu()).numpy().ravel()
pred_price = np.expm1(pred_scaled * checkpoint['y_std'] + checkpoint['y_mean'])

print(f'{best_name}: RMSE={root_mean_squared_error(y_test, pred_price):.0f}, R2={r2_score(y_test, pred_price):.4f}')

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(y_test, pred_price, alpha=0.3, s=10)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', label='Идеальный прогноз')
plt.xlabel('Реальная цена')
plt.ylabel('Предсказанная цена')
plt.title('Предсказание vs реальность')
plt.legend()
plt.show()